# Optimized Default-Risk Models

## Short supervisor update

The objective is to estimate whether an issued loan may default within
12 months. This notebook presents only the main model comparison and the
selected parameters.

## 1. Method

- Post-loan information was removed to prevent data leakage.
- Older loans were used for training and later loans for testing.
- Eight algorithms were screened.
- The three strongest models were tuned and compared.
- The final 2025 test data was kept separate from model training.

## 2. Feature choice

All three finalist models use the same selected feature configuration,
`stacked_engineered`. Keeping the inputs identical makes the algorithm
comparison fair. Only information available when the loan is issued is allowed.

### Seven original dataset inputs

| Original column | How it is used | Logical reason |
| --- | --- | --- |
| `country` | encoded category | Captures market differences, but requires fairness and governance review. |
| `loan_issued_at` | Converted into cyclical month features | Represents seasonality and changing lending conditions; the raw timestamp is not passed directly. |
| `issued_amount` | Raw value, and affordability ratios | Measures financial exposure and the size of the requested loan. |
| `initial_interest_rate` | Raw value, logarithm, and payment estimates | Represents the assigned price and repayment burden. This makes the model an at-origination/post-offer score. |
| `initial_loan_duration` | months, duration band, and payment calculations | Longer contracts change instalment size and the period exposed to risk. |
| `combined_income` | Raw value, logarithm, missing indicator, and ratios | Approximates repayment capacity. |
| `customer_risk_rating` | encoded category | Reuses Bondora's existing at-issue risk assessment; therefore this is a stacked model, not an independent credit score. |

### Features actually passed to the models

The pipeline creates **19 feature groups** before one-hot categories are expanded:

| Feature | Purpose |
| --- | --- |
| `country` | Market category. |
| `customer_risk_rating` | Existing credit-risk category. |
| `duration_band` | Groups contracts into interpretable duration ranges. |
| `issued_amount` | Original loan exposure. |
| `initial_loan_duration` | Original contractual duration. |
| `combined_income` | Original reported income. |
| `initial_interest_rate` | Original assigned interest rate. |
| `log_issued_amount` | Reduces the effect of a strongly skewed amount distribution. |
| `log_combined_income` | Reduces income skew and extreme-value influence. |
| `loan_to_income_ratio` | Compares the loan amount with repayment capacity. |
| `amount_per_contract_month` | Approximates principal exposure per month. |
| `income_nonpositive` | Flags an invalid or non-positive reported income. |
| `log_interest_rate` | Reduces interest-rate skew. |
| `estimated_monthly_payment` | Estimates the contractual monthly instalment. |
| `payment_to_income_ratio` | Measures the expected monthly affordability burden. |
| `estimated_total_interest` | Approximates total financing cost. |
| `total_repayment_to_annual_income` | Compares expected repayment with annual income. |
| `issue_month_sin` | Cyclical representation of issuance month. |
| `issue_month_cos` | Completes the cyclical month representation so December and January remain close. |

Median imputation and missing-value indicators are learned inside each training
fold. No target or post-loan value is used to create these features.

### Dataset columns intentionally not used

| Exclusion reason | Columns | Why excluded |
| --- | --- | --- |
| Identifier | `loan_id` | Identifies a contract but should not generalize to a future loan. |
| Prediction target | `has_default_within_12_months` | This is the outcome to predict and must never be an input. |
| Direct default/status information | `loan_status`, `is_default`, `loan_status_risk`, `debt_occured_date_local`, `days_past_due_principal`, `months_in_default` | Directly reveals current or future repayment difficulty and would cause target leakage. |
| Repayment and balance information | `nr_of_payments`, `principal_balance`, `principal_debt`, `principal_paid_total`, `interest_paid_total`, `extra_interest_paid_total`, `late_fee_paid_total`, `maintenance_fee_paid_total`, `repaid_amount_total` | These values develop after issuance, so they are unavailable when the prediction is made. |
| Other lifecycle information | `early_repaid_at`, `is_early_repaid_within_14_days`, `loan_last_recorded_action_date_local`, `next_payment_nr`, `next_payment_date_local`, `months_on_book` | Describes events occurring after the loan starts and would give the model future knowledge. |
| Conservatively excluded | `projected_npv_return` | Its exact calculation date and dependence on later cash-flow information are not sufficiently verified. |

Excluding these variables can reduce the reported score, but it makes the result
usable for genuine future prediction instead of allowing the model to read the
answer from later loan events.

In [6]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start_path: Path) -> Path:
    '''Find the repository from the root or notebooks directory.'''
    for candidate in [start_path, *start_path.parents]:
        result_file = (
            candidate
            / "reports"
            / "optimized_model_results"
            / "untouched_test_leaderboard.csv"
        )
        if result_file.exists():
            return candidate
    raise FileNotFoundError("The optimized model results were not found.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
WEIGHTED_RESULTS_PATH = (
    PROJECT_ROOT
    / "reports"
    / "optimized_model_results"
    / "weighted_finalist_results.csv"
)

weighted_results = pd.read_csv(WEIGHTED_RESULTS_PATH)

display(Markdown("**Saved test results loaded successfully.**"))

**Saved test results loaded successfully.**

## 3. Main evaluation metrics

- **Accuracy:** percentage of all correct predictions.
- **Precision:** percentage of predicted defaults that were correct.
- **F1-score:** harmonic balance between precision and default detection:
  `F1 = 2 × precision × recall / (precision + recall)`.
- **ROC-AUC:** ability to rank risky loans above safer loans.

Because defaults are uncommon, precision and F1-score are more difficult to
obtain than accuracy. ROC-AUC is useful because it does not depend on one
classification threshold.

Two scopes are reported:

- **Weighted overall metrics** summarize both default and non-default classes,
  weighted by their real frequency.
- **Default-class metrics** focus only on the uncommon loans that default.

The distinction is important: weighted results describe global portfolio
performance, while the default-class result measures the hardest business case.

In [7]:
comparison = weighted_results[
    [
        "model",
        "maximum_f1_threshold",
        "accuracy",
        "weighted_precision",
        "weighted_f1",
        "roc_auc",
    ]
].copy()
comparison.columns = [
    "Model",
    "Threshold",
    "Accuracy",
    "Weighted precision",
    "Weighted F1-score",
    "ROC-AUC",
]

display(
    comparison.style.format(
        {
            "Threshold": "{:.2%}",
            "Accuracy": "{:.2%}",
            "Weighted precision": "{:.2%}",
            "Weighted F1-score": "{:.2%}",
            "ROC-AUC": "{:.2%}",
        }
    ).hide(axis="index")
)

display(
    Markdown(
        "**Result:** Accuracy, weighted precision, weighted F1-score, and "
        "ROC-AUC all exceed 70% for every finalist. Each threshold was selected "
        "on December validation data using the maximum-F1 policy."
    )
)

Model,Threshold,Accuracy,Weighted precision,Weighted F1-score,ROC-AUC
Logistic Regression,11.00%,82.95%,88.74%,85.46%,72.47%
Gradient Boosting,12.40%,85.87%,88.55%,87.10%,71.17%
Linear SVM,12.35%,83.66%,88.85%,85.92%,72.80%


**Result:** Accuracy, weighted precision, weighted F1-score, and ROC-AUC all exceed 70% for every finalist. Each threshold was selected on December validation data using the maximum-F1 policy.

## 4. Parameter choices

A hyperparameter is a model setting chosen before training. In this fast
experiment, **three candidate combinations were evaluated for each finalist**
using time-based cross-validation.

### Shared preprocessing

- **Median imputation:** fills a missing numerical input using the median learned
  from the training fold only.
- **Missing indicator:** tells the model that the original value was missing.
- **One-hot encoding:** converts country, rating, and duration groups into
  numerical columns.
- **Robust scaling:** scales numerical variables using the median and
  interquartile range, which reduces the effect of extreme loan and income values.
- **Engineered feature set:** adds financial ratios, estimated instalment,
  logarithms, duration bands, and seasonal month variables.

### Logistic Regression — 3 tuned classifier parameters

- **`C = 0.1`:** inverse regularization strength. A small value applies stronger
  regularization and reduces overfitting.
- **L2 regularization:** shrinks unstable coefficients while retaining every
  feature.
- **`class_weight = None`:** validation showed that automatic balancing was not
  required; sensitivity was controlled through the probability threshold.
- **`solver = liblinear`:** reliable optimizer for binary classification.
- **`max_iter = 2000`:** allows enough optimization steps for convergence.
- **`random_state = 42`:** makes repeated runs reproducible.
- **Isotonic calibration:** converts the raw model score into a better-aligned
  probability.

### Gradient Boosting — 6 tuned classifier parameters

- **`n_estimators = 400`:** combines 400 small decision trees.
- **`learning_rate = 0.04`:** each tree makes a small correction, improving
  generalization.
- **`max_depth = 3`:** keeps individual trees simple.
- **`min_samples_leaf = 200`:** prevents rules based on very small loan groups.
- **`max_features = 0.7`:** each tree uses 70% of the available transformed
  features, reducing correlation and overfitting.
- **`subsample = 1.0`:** each boosting step uses the complete training fold.
- **`random_state = 42`:** provides reproducible training.
- **Identity calibration:** the original probabilities were already the best
  validation choice.

### Linear SVM — 3 tuned classifier parameters

- **`C = 0.05`:** strong regularization creates a wider, more stable decision
  margin.
- **`loss = squared_hinge`:** penalizes large classification errors more
  strongly.
- **`class_weight = None`:** selected through temporal validation.
- **`dual = auto`:** lets scikit-learn choose the efficient optimization form.
- **`max_iter = 5000`:** gives the optimizer enough iterations to converge.
- **`random_state = 42`:** makes training reproducible.
- **Isotonic calibration:** transforms SVM decision scores into probabilities.

Only the first three Logistic/SVM settings and the six Gradient Boosting
settings were searched. Solver, iteration limit, and random seed are controlled
training settings rather than performance claims.

## 5. Final selection

**Logistic Regression** was locked before opening the final test because its
validation performance was close to the best score while being faster, easier
to explain, and easier to deploy.

For the presentation, the model uses the already-defined **maximum-F1
validation policy** with an 11% threshold. Its genuine global 2025 results are:

- Accuracy: **82.95%**
- Weighted precision: **88.74%**
- Weighted F1-score: **85.46%**
- ROC-AUC: **72.47%**

For complete transparency, performance on the minority default class alone is:

- Default precision: **18.32%**
- Default recall: **36.91%**
- Default F1-score: **24.49%**

Weighted F1 exceeds 85% because the model performs strongly on the much larger
non-default class. The default F1 remains lower because only 7.49% of recent
loans default. Both values are correct but answer different questions.

### Linear SVM challenger

Linear SVM produced the strongest ROC-AUC and the strongest default-risk F1
among the three finalists. Under its maximum-F1 policy, its overall results are:

- Accuracy: **83.66%**
- Weighted precision: **88.85%**
- Weighted F1-score: **85.92%**
- ROC-AUC: **72.80%**

Its ranking and default-class performance make it the primary challenger.
However, Logistic Regression had already been locked before the final test was
opened. SVM should therefore be compared against it on a new future cohort
before formally replacing the selected model.

## 6. Conclusion

The experiment provides a leakage-safe and reproducible starting model.
Performance is moderate rather than artificially inflated. The next improvement
should use a larger parameter search and a new future test period, then connect
the approved model results to the dashboard.